<a href="https://colab.research.google.com/github/reza-abdolee/Generative-AI-Systems---Winter-2027/blob/main/Module_00_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 00 Lab: Setup and Introduction to Google Colab and the NRP LLM Service
**COMP 490 Generative AI Systems · CSU Channel Islands · Dr. Reza Abdolee**

This notebook follows the Module 00 lab manual. Run the cells **from top to bottom, in order**: click a cell and press **Shift + Enter**.

**Before you start** (Part 3 of the manual):
1. Open the NRP join link your instructor posted on Canvas and log in with your CSUCI credentials.
2. Create an API key at [nrp.ai/llmtoken](https://nrp.ai/llmtoken/) and copy it.
3. In this notebook, click the 🔑 **Secrets** icon on the left, add a secret named `NRP_API_KEY`, paste your key as the value, and turn on **Notebook access**.

> Never paste your API key into a code cell. This notebook reads it from Colab Secrets.

## Part 2 – Your First Code Cell
Run the cell below. If you see two lines of output, your notebook is working.

In [1]:
print("Hello from Google Colab!")
print("If you can see this message, your notebook is working.")

Hello from Google Colab!
If you can see this message, your notebook is working.


## Part 4 – Check That Your API Key Is Stored in Colab Secrets
This should print **True** (not the key itself). If it prints False or shows an error, check the secret's name and the Notebook access toggle.

In [2]:
# Test that your secret is available.
# This should print True, not the key itself.
from google.colab import userdata
api_key = userdata.get("NRP_API_KEY")
print(api_key is not None)

True


**Optional:** test the raw connection to the NRP with the `curl` command-line tool. `HTTP status: 200` means your key works; `401` means the key is wrong or expired.

In [3]:
# Optional: test the raw connection with curl.
# "HTTP status: 200" means your key works; 401 means the key is wrong or expired.
import os
from google.colab import userdata
os.environ["NRP_AUTH"] = "Authorization: Bearer " + userdata.get("NRP_API_KEY").strip()
os.environ["NRP_URL"] = "https://ellm.nrp-nautilus.io/v1/models"
!curl -s -o /dev/null -w "HTTP status: %{http_code}\n" -H "$NRP_AUTH" $NRP_URL

HTTP status: 200


## Part 5 – Install Required Libraries
The exclamation mark runs a command in the notebook environment. The NRP uses the OpenAI-compatible API format, so the labs use the `openai` Python package.

In [4]:
!pip install -q openai pandas

## Part 6 – Make Your First NRP API Call
This connects to the NRP, lists the models available right now, and sends one short prompt.

**About model "thinking":** every NRP chat model thinks privately before it answers, and that thinking uses tokens. `gpt-oss` cannot turn thinking off, so this call uses `max_tokens=500` to leave room for both the thinking and the answer.

In [5]:
from google.colab import userdata
from openai import OpenAI

# The NRP LLM service uses the OpenAI-compatible API format.
client = OpenAI(
    api_key=userdata.get("NRP_API_KEY").strip(),
    base_url="https://ellm.nrp-nautilus.io/v1",
)

# Models used in this course. If a model is retired, change it here only.
MODELS = {
    "main": "gpt-oss",   # OpenAI gpt-oss-120b; thinking is always on
    "fast": "gemma",     # Google Gemma 4 31B; thinking can be turned off
}

# List the models that are available right now.
print([m.id for m in client.models.list().data])

response = client.chat.completions.create(
    model=MODELS["main"],
    messages=[
        {"role": "user", "content": "Explain what an API is in one sentence."}
    ],
    temperature=0.2,
    max_tokens=500,                             # room for hidden thinking + answer
    extra_body={"reasoning_effort": "low"},     # gpt-oss accepts low, medium, high
)
print(response.choices[0].message.content)

['gemma-small', 'gemma4-small', 'gemma4-12b', 'agents', 'coding', 'general', 'cuda', 'kubernetes', 'long-context', 'reasoning', 'vision', 'qwen3-mc', 'gemma-mc', 'gpt-oss', 'gemma', 'gemma-vl', 'kimi', 'minimax', 'minimax-m2', 'deepseek-v4-flash', 'gemma-small-e4b', 'glm-5', 'qwen3', 'qwen3-vl', 'qwen3-embedding', 'qwen3-small', 'service']
An API (Application Programming Interface) is a set of defined rules and protocols that allows different software applications to communicate with each other by exposing specific functions, data formats, and commands for developers to use.


## Part 7 – Measure Latency and Token Usage
`run_simple_test()` records response time and token usage for one request. It uses the fast model (`gemma`) with thinking turned off, and retries automatically if the shared service is briefly busy.

In [6]:
import time
import pandas as pd
from openai import BadRequestError

def run_simple_test(prompt, model=MODELS["fast"], temperature=0.2,
                    reasoning_effort="none", max_tokens=300, retries=3):
    # Run one NRP request and return engineering metrics.
    # Retries temporary failures (busy model, rate limit, maintenance).
    for attempt in range(1, retries + 1):
        try:
            start_time = time.time()
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
                extra_body={"reasoning_effort": reasoning_effort},
            )
            latency = time.time() - start_time
            break
        except BadRequestError:
            raise   # HTTP 400: a setting is wrong, so retrying will not help
        except Exception as error:
            if attempt == retries:
                raise
            wait = 10 * attempt
            print(f"Attempt {attempt} failed ({error}). Retrying in {wait} s...")
            time.sleep(wait)

    return {
        "model": model,
        "temperature": temperature,
        "reasoning_effort": reasoning_effort,
        "prompt": prompt,
        "response": response.choices[0].message.content,
        "latency_seconds": round(latency, 2),
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens
    }

result = run_simple_test("Give one reason engineers measure latency.")
result

{'model': 'gemma',
 'temperature': 0.2,
 'reasoning_effort': 'none',
 'prompt': 'Give one reason engineers measure latency.',
 'response': 'Engineers measure latency to **ensure a positive user experience**, as high latency (lag) can make an application feel unresponsive or unusable.',
 'latency_seconds': 1.29,
 'prompt_tokens': 20,
 'completion_tokens': 28,
 'total_tokens': 48}

## Part 8 – Mini Experiment: Temperature and Consistency
Runs the same prompt 3 times at temperature 0.0 and 3 times at 0.8.

**Observation questions:**
- Were the temperature 0.0 responses more similar to each other?
- Were the temperature 0.8 responses more varied?
- Did token usage or latency change across runs?

In [7]:
import time

prompt = "Write one sentence explaining why AI systems need testing."
experiments = []

for temp in [0.0, 0.8]:
    for run in range(1, 4):
        output = run_simple_test(prompt, temperature=temp)
        output["run"] = run
        experiments.append(output)
        time.sleep(2)  # Shares the NRP service fairly with other users.

experiment_df = pd.DataFrame(experiments)
experiment_df[["run", "temperature", "latency_seconds",
               "completion_tokens", "total_tokens", "response"]]

,run,temperature,latency_seconds,completion_tokens,total_tokens,response
0,1,0.0,0.58,31,54,AI systems require testing to ensure they are ...
1,2,0.0,0.54,31,54,AI systems require testing to ensure they are ...
2,3,0.0,0.55,31,54,AI systems require testing to ensure they are ...
3,1,0.8,0.51,28,51,"AI systems require testing to identify biases,..."
4,2,0.8,0.53,30,53,AI systems require testing to ensure they are ...
5,3,0.8,0.83,30,53,AI systems require testing to ensure they are ...


**Optional extension:** the same prompt on `gpt-oss`, which always thinks. Why does it report more completion tokens for a similar one-sentence answer, and what does that mean for cost and speed?

In [8]:
# Optional extension: the same prompt on a model that always thinks.
thinking = run_simple_test(prompt, model=MODELS["main"],
                           reasoning_effort="low", max_tokens=500)
print("gemma (thinking off) completion tokens:", experiments[0]["completion_tokens"])
print("gpt-oss (thinking low) completion tokens:", thinking["completion_tokens"])

gemma (thinking off) completion tokens: 31
gpt-oss (thinking low) completion tokens: 42


## Part 9 – Save Your Results to CSV
After running the cell: click the 📁 folder icon on the left, find `module00_setup_results.csv`, click the three dots next to it, and choose **Download**.

In [9]:
experiment_df.to_csv("module00_setup_results.csv", index=False)
print("Saved module00_setup_results.csv")

Saved module00_setup_results.csv


## Part 10 – If Something Goes Wrong
| Problem | Fix |
|---|---|
| **401** authentication error | Check that the secret is named `NRP_API_KEY` and the key is correct (from nrp.ai/llmtoken). Your course access may also have ended. |
| **400** Bad Request | A setting the model does not accept, usually `reasoning_effort`. gpt-oss: low/medium/high; gemma: none. |
| **404** or "model not found" | Print the model list (Part 6) and update the `MODELS` dictionary. |
| `ModuleNotFoundError: openai` | Run the Part 5 install cell again. |
| **429** rate limit | Wait 60 seconds; keep `time.sleep(2)` or longer in loops. |
| Timeout, 502, or 503 | The shared service is busy; `run_simple_test` retries automatically. Try again later if it keeps failing. |
| Empty response from a thinking model | Increase `max_tokens` or lower `reasoning_effort`. |
| No output appears | Choose Runtime → Restart session, then run the cells from the top in order. |

## Before you submit
- [ ] Every cell above ran without errors.
- [ ] `module00_setup_results.csv` is downloaded.
- [ ] This notebook does **not** contain your API key.
- [ ] Submit this notebook (Share link or .ipynb) with your short setup reflection.